In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [25]:
rng = np.random.default_rng(42)

In [54]:
# population_alphas - (pop_size, K)
def fitness_pop(population_alphas, lengths, target_point):
    thetas = np.cumsum(population_alphas, axis=1)

    tip_xs = np.sum(lengths * np.cos(thetas), axis=1)
    tip_ys = np.sum(lengths * np.sin(thetas), axis=1)

    dist_sq = (tip_xs - target_point[0])**2 + (tip_ys - target_point[1])**2

    return dist_sq

def init_population(pop_size, dim, bounds, rng, init_sigma=0.1):
    low, high = bounds
    x = rng.uniform(low, high, size=(pop_size, dim))
    sigma0 = init_sigma * (high - low)
    sigma = np.ones((pop_size, dim)) * sigma0
    return x, sigma

def mutate(pop_x, sigma, tau, tau0, rng, min_sigma=1e-8):
    n, d = pop_x.shape

    eps0 = rng.normal(loc=0.0, scale=tau0, size=(n, 1))
    eps = rng.normal(loc=0.0, scale=tau, size=(n, d))

    new_sigma = sigma * np.exp(eps + eps0)
    new_sigma = np.maximum(new_sigma, min_sigma)

    noise = rng.normal(loc=0.0, scale=new_sigma)
    new_x = pop_x + noise
    return new_x, new_sigma


def es(
        lengths,
        target_point,
        rng,
        bounds: np.ndarray,
        mu: int = 15,
        lambda_: int = 100,
        iters: int = 1000,
        plus: bool = True,
        tau: float | None = None,
        tau0: float | None = None,
):
    dim = len(lengths)

    if tau is None:
        tau = 1.0 / np.sqrt(2.0 * np.sqrt(dim))
    if tau0 is None:
        tau0 = 1.0 / np.sqrt(2.0 * dim)

    x, sigma = init_population(mu, dim, bounds, rng)
    fitness = fitness_pop(x, lengths, target_point)

    history = []
    history_solutions = []
    low, high = bounds

    for _ in range(iters):
        parent_indices = rng.integers(0, mu, size=lambda_)
        parent_x = x[parent_indices]
        parent_sigma = sigma[parent_indices]

        child_x, child_sigma = mutate(parent_x, parent_sigma, tau, tau0, rng)
        child_x = np.clip(child_x, low, high)
        child_fitness = fitness_pop(child_x, lengths, target_point)

        if plus:
            all_x = np.vstack([x, child_x])
            all_sigma = np.vstack([sigma, child_sigma])
            all_fitness = np.concatenate([fitness, child_fitness])

            best_idx = np.argsort(all_fitness)[:mu]
            x = all_x[best_idx]
            sigma = all_sigma[best_idx]
            fitness = all_fitness[best_idx]
        else:
            best_idx = np.argsort(child_fitness)[:mu]
            x = child_x[best_idx]
            sigma = child_sigma[best_idx]
            fitness = child_fitness[best_idx]

        history.append(fitness[0])
        history_solutions.append(x[0].copy())

    best_idx = np.argmin(fitness)
    return x[best_idx], fitness[best_idx], np.array(history), np.array(history_solutions)


In [85]:
def get_joint_coordinates(alphas, lengths):
    thetas = np.cumsum(alphas)

    x_coords = np.concatenate(([0], np.cumsum(lengths * np.cos(thetas))))
    y_coords = np.concatenate(([0], np.cumsum(lengths * np.sin(thetas))))

    return x_coords, y_coords

import ipywidgets as widgets

def plot_robot_arm(gen_idx, lengths, history_solutions, target_point ):
    alphas = history_solutions[gen_idx]
    x_coords, y_coords = get_joint_coordinates(alphas, lengths)
    tip_x, tip_y = x_coords[-1], y_coords[-1]

    dist = np.sqrt((tip_x - target_point[0])**2 + (tip_y - target_point[1])**2)
    plt.figure(figsize=(10, 10))

    max_reach = np.sum(lengths)
    limit = np.max([max_reach * 1.2, abs(target_point[0]), abs(target_point[1])]) + 0.5
    plt.xlim(-limit, limit)
    plt.ylim(-limit, limit)


    plt.plot(x_coords, y_coords, '-o', linewidth=3, label='Robot')
    plt.scatter(target_point[0], target_point[1], color='red', s=150, marker='x')
    plt.grid(True, which='both', linestyle='--')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.title(f"{"Inverse Kinematic Problem"}\nGeneration{gen_idx}\nError: {dist:.4f}")
    plt.show()


In [86]:
lengths = np.array([1., 2., 1.5])
target_point = np.array([1, 3.])
bounds = np.array([[-np.pi, -np.pi/4, -np.pi/2], [np.pi, np.pi/4, np.pi/2,]])

alphas, alphas_fitness, history, history_solutions = es(lengths=lengths, target_point=target_point, rng=rng, bounds=bounds, iters=40)

widgets.interact(
    plot_robot_arm,
    gen_idx=widgets.IntSlider(min=0, max=len(history_solutions)-1, value=0),
    history_solutions=widgets.fixed(history_solutions),
    lengths=widgets.fixed(lengths),
    target_point=widgets.fixed(target_point)
);

interactive(children=(IntSlider(value=0, description='gen_idx', max=39), Output()), _dom_classes=('widget-inte…